# HR Sourcing Specialist Agent Loop

An agent loop simulating an HR sourcing pipeline:
1. Sourcing talents
2. Qualifying the talent
3. Outreach
4. Hand-off to hiring manager

In [3]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)
openai = OpenAI()

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [6]:
# Pipeline State
candidates = {}

def get_pipeline_report() -> str:
    if not candidates:
        result = "Pipeline is empty.\n"
    else:
        result = "Current Pipeline:\n"
        for name, info in candidates.items():
            result += f"- {name}: Stage=[{info['stage']}], Notes=[{info['notes']}]\n"
    show(result)
    return result

def add_candidates(names: list[str]) -> str:
    for name in names:
        if name not in candidates:
            candidates[name] = {"stage": "Sourced", "notes": "Newly sourced"}
    show(f"[bold green]Added {len(names)} candidate(s)[/bold green]")
    return get_pipeline_report()

def update_candidate_stage(name: str, new_stage: str, notes: str) -> str:
    if name in candidates:
        candidates[name]["stage"] = new_stage
        candidates[name]["notes"] = notes
        show(f"[bold blue]Updated {name}[/bold blue] to '{new_stage}'. Notes: {notes}")
    else:
        return f"Candidate {name} not found."
    return get_pipeline_report()

In [7]:
add_candidates_json = {
    "name": "add_candidates",
    "description": "Add newly sourced candidates to the pipeline.",
    "parameters": {
        "type": "object",
        "properties": {
            "names": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of candidate names to add"
            }
        },
        "required": ["names"],
        "additionalProperties": False
    }
}

update_candidate_stage_json = {
    "name": "update_candidate_stage",
    "description": "Update the pipeline stage for a candidate (e.g., 'Qualified', 'Outreach', 'Handed-off').",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "Name of the candidate"
            },
            "new_stage": {
                "type": "string",
                "description": "The new pipeline stage (e.g. 'Qualified', 'Outreach', 'Handed-off', 'Rejected')"
            },
            "notes": {
                "type": "string",
                "description": "Details about the update or interaction"
            }
        },
        "required": ["name", "new_stage", "notes"],
        "additionalProperties": False
    }
}

get_pipeline_report_json = {
    "name": "get_pipeline_report",
    "description": "Get the current report of all candidates and their stages in the pipeline.",
    "parameters": {
        "type": "object",
        "properties": {},
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": add_candidates_json},
    {"type": "function", "function": update_candidate_stage_json},
    {"type": "function", "function": get_pipeline_report_json}
]

In [8]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

In [9]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-5.2",
            messages=messages,
            tools=tools,
            reasoning_effort="none"
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [10]:
system_message = """
You are an HR Sourcing Specialist Agent. Your job is to manage the candidate sourcing pipeline.
The pipeline stages are:
1. Sourced
2. Qualified
3. Outreach
4. Handed-off

When given a task, use your tools to construct your pipeline by acting on the candidates.
Provide your final solution and pipeline summary in Rich console markup.
Do not ask the user questions; respond only with the answer after using your tools.
"""

user_message = """
We have a new open role for a Senior Software Engineer.
I found three potential candidates: Alice, Bob, and Charlie.
Please add them to the pipeline.

Then, simulate their progression:
- Alice and Charlie are 'Qualified' because they have python experience. Bob is 'Rejected' (only Java).
- We do 'Outreach' to Alice and Charlie. 
- Alice replies positively, Charlie doesn't reply.
- Finally, 'Handed-off' Alice to the hiring manager.

Process this pipeline using your tools.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

In [ ]:
candidates = {}
loop(messages)